# Chapter 13 — Transforms are rules

Step 12: the first IR-to-IR machinery — **vmap** and **jvp**, plus two
user-directed additions: the in-kernel derivative operator **`D`** (GLSL's
`dFdx` idea, done analytically) and **named matrix contraction** with
batching as the payoff. Design: `docs/design/110_transforms-and-derivatives.md`.

The architecture priced transforms carefully and demanded a spike before
commitment (>350 counted lines ⇒ re-hear). The spike's finding reshaped the
step and is worth stating up front: **our vmap is SIMT-shaped, not
SIMD-shaped**. It does not widen values with a batch dimension (JAX's way,
where batched predicates force execute-both-and-select and break lazy
branches); it adds one *coordinate parameter* and weaves it into the
accesses that need it — the same move the compute family made when params
became thread coordinates. Intermediates stay scalar; each lane runs its
own branches and trip counts; `if`/`for` need ZERO new machinery. The
transform satellite landed at 338 counted lines (after the review pass hardened it) — under the threshold, no re-hear.

In [1]:
import numpy as np

import pdum.dsl  # noqa: F401
from pdum.dsl import jit, no_compile
from pdum.dsl.kernel.registry import DEFAULT
from pdum.dsl.stdlib.arrays import Named
from pdum.dsl.stdlib.transforms import jvp, vmap

## jvp: dual numbers, compiled

Forward-mode AD: every value `a` gets a shadow `ȧ`, every primitive a
two-line linearization rule (`mul`: `aḃ + ȧb`; `sqrt`: `ṫ/2√a`). No tape,
no tracing — the tangent chain is just MORE NODES in the same region,
synthesized once at build. `jvp(f)(*args, *tangents)` returns
`(primal, directional derivative)`. Control flow: branches get a parallel
lazy `core.if` on the same condition; loops WIDEN — the carry becomes
`(primal, tangent)` because the tangent recurrence needs the primal carry
each iteration:

In [2]:
def make(c):
    @jit()
    def f(x):
        y = c * x * x + sqrt(x)  # noqa: F821
        if y > 2.0:
            y = y * 2.0
        acc = y
        for i in range(3):
            acc = acc * x
        return acc

    return f


f = make(0.5)
jf = jvp(f)
x0, eps = 1.3, 1e-7
p, t = jf(x0, 1.0)
fd = (f(x0 + eps) - f(x0 - eps)) / (2 * eps)
print(f"primal      : {p:.12f}   f(x): {f(x0):.12f}")
print(f"jvp tangent : {t:.8f}   finite diff: {fd:.8f}   |Δ| = {abs(t - fd):.1e}")
c0 = DEFAULT.specializations.compiles
with no_compile():  # a fresh closure UNDER the transform: still a hit
    jvp(make(0.9))(2.0, 1.0)
print("fresh closure under jvp -> compiles:", DEFAULT.specializations.compiles - c0)

primal      : 4.361430408943   f(x): 4.361430408943
jvp tangent : 13.88438764   finite diff: 13.88438765   |Δ| = 9.0e-09
fresh closure under jvp -> compiles: 0


In [3]:
key = DEFAULT.specializations.key_for(
    jf, tuple(DEFAULT.table.fingerprint(a) for a in (x0, 1.0)), DEFAULT.backend_for(jf.kind).fp
)
print(DEFAULT.specializations.probe(key).artifact.__pdum_source__)

import math
from struct import unpack_from as _u
from pdum.dsl.kernel.registry import Out as _Out

def _tdiv(a, b):  # exact trunc division (numeric policy: C semantics)
    q = a // b
    return q + 1 if q < 0 and q * b != a else q

def _tmod(a, b):
    return a - _tdiv(a, b) * b

def kernel(staging, leaves):
    if any(isinstance(x, _Out) for x in leaves):
        raise TypeError("the python backend takes no launcher data (out= is for device targets)")
    v0 = 0
    v1 = 3
    v2 = _u('<d', staging, 0)[0]
    v3 = _u('<d', staging, 8)[0]
    v4 = v2 * v3
    v5 = v4 * v3
    v6 = math.sqrt(v3)
    v7 = v5 + v6
    v8 = 2.0
    v9 = v7 > v8
    v10 = 2.0
    if v9:
        v11 = v7 * v10
        v12 = v11
    else:
        v12 = v7
    v13 = _u('<d', staging, 16)[0]
    v14 = v4 * v13
    v15 = v2 * v13
    v16 = v15 * v3
    v17 = v14 + v16
    v18 = 2.0
    v19 = v18 * v6
    v20 = v13 / v19
    v21 = v17 + v20
    if v9:
        v22 = v21 * v10
        v23 = v22
    else:
        

Read the render: ONE loop carrying a `(primal, tangent)` pair, the product
rule inlined in its body, and a `(v.., v..)` pair returned at the end. The
`Derived("jvp", …)` identity means the transform participates in the
two-tier cache like any kernel — the fresh-closure hit above is the ch03
thesis surviving its first transform.

## `D`: the derivative of anything, with respect to *here*

The user-directed operator (110 §3): `D(x)` = the partials of any
intermediate w.r.t. the ENCLOSING KERNEL'S parameters — one per param,
positionally. In the shader world where params ARE pixel coordinates, this
is GLSL's `dFdx`/`dFdy` — except GLSL computes a finite difference across
the 2×2 pixel quad, while `D` differentiates analytically. And compute
shaders have NO quads (WGSL has no `dpdx` outside fragment stage), so
analytic `D` is not a convenience there — it is the only derivative in
town, and it is exact:

In [4]:
def make_ring(cx, cy, r):
    @jit()
    def k(i, j):
        d = sqrt((i - cx) * (i - cx) + (j - cy) * (j - cy)) - r  # noqa: F821
        di, dj = D(d)  # noqa: F821 — analytic screen-space gradient
        return abs(di) + abs(dj)  # noqa: F821 — this IS fwidth(d)

    return k


k = make_ring(0.0, 0.0, 1.0)
print("gradient magnitude proxy at (3,4):", f"{k(3.0, 4.0):.6f}")
print("  (|3/5| + |4/5| = 1.4 — the SDF's unit gradient, split across axes)")


# Structure differentiates structurally: D of a tuple is a tuple per param.
@jit()
def kt(i, j):
    p = (i * j, i + j)
    dp_i, dp_j = D(p)  # noqa: F821
    return dp_i[0] + dp_j[1]  # d(ij)/di + d(i+j)/dj


print("D of a tuple:", kt(3.0, 5.0), " (j + 1 =", 5.0 + 1.0, ")")

gradient magnitude proxy at (3,4): 1.400000
  (|3/5| + |4/5| = 1.4 — the SDF's unit gradient, split across axes)
D of a tuple: 6.0  (j + 1 = 6.0 )


### Analytic anti-aliasing (the classic, minus the quads)

The canonical `fwidth` trick: smooth a hard edge over exactly one pixel's
worth of parameter change, whatever the zoom. `demo.graphics` ships the GL
vocabulary (`ddx`/`ddy`/`fwidth`) as one-line batteries over `D` — an
explicit import, per the stdlib-minimalism rule:

In [5]:
from pdum.dsl.demo import graphics  # noqa: F401 — wires ddx/ddy/fwidth (+ Color) into DEFAULT


def make_disk(cx, cy, r, zoom):
    @jit()
    def mask(i, j):
        d = length2(((i - 32.0) * zoom - cx, (j - 32.0) * zoom - cy)) - r  # noqa: F821
        w = fwidth(d)  # noqa: F821 — one pixel's worth of d, ANALYTICALLY
        return 1.0 - smoothstep(-w, w, d)  # noqa: F821

    return mask


for zoom in (0.05, 0.2):
    mask = make_disk(0.0, 0.0, 1.0, zoom)
    shades = " ░▒▓█"
    print(f"zoom {zoom}: edge stays one pixel wide")
    for row in range(8, 24, 2):
        line = "".join(shades[min(4, int(mask(float(row), float(col)) * 4.99))] for col in range(16, 48))
        print("   ", line)

zoom 0.05: edge stays one pixel wide
                                    
                                    
                ░░▒▒▒▒▒░░           
           ▒▓███████████████▓▒      
        ▒▓█████████████████████▓▒   
     ░▓███████████████████████████▓░
    ▒███████████████████████████████
    ████████████████████████████████
zoom 0.2: edge stays one pixel wide
                                    
                                    
                                    
                                    
                                    
                                    
                                    
                                    


The edge is exactly one pixel of smooth ramp at BOTH zooms — that is
`fwidth` adapting the smoothing width to the local derivative, computed
exactly, with no quad neighbors, on a plain CPU backend. Kernels that never
call `D` pay nothing: the tangent slice is synthesized only where demanded
(tier 2 can verify — D-free kernels mint the same artifacts they always
did).

## vmap: written ignorant, batched by declaration

The batching-ignorance arc. A kernel written for ONE element, against data
whose batch axis it never mentions — the pedantic axis checking REFUSES it
(an unaccounted axis is an error, not a silent broadcast). `vmap` is the
declaration that accounts for it:

In [6]:
data = Named(np.arange(12.0).reshape(3, 4), ("batch", "x"))


def make_sum(t):
    @jit()
    def g(k):
        return t.isel(x=k) * 10.0  # batch-IGNORANT: only 'x' is named

    return g


g = make_sum(data)
try:
    g(1)
except Exception as e:
    print("🚫 without vmap:", str(e)[:95], "…")

vg = vmap(g, axis="batch")  # the declaration: 'batch' is mine now
print("lanes:", [vg(1, b) for b in range(3)])
c0 = DEFAULT.specializations.compiles
with no_compile():  # 100 lanes instead of 3: same TYPE, zero recompiles
    vmap(make_sum(Named(np.ones((100, 4)), ("batch", "x"))), axis="batch")(1, 99)
print("new batch size -> compiles:", DEFAULT.specializations.compiles - c0)

🚫 without vmap: isel is pedantic on purpose: name every axis exactly once as a keyword — expected {'x', 'batch' …
lanes: [np.float64(10.0), np.float64(50.0), np.float64(90.0)]
new batch size -> compiles: 0


In [7]:
key = vg, (DEFAULT.table.fingerprint(1), DEFAULT.table.fingerprint(0))
k2 = DEFAULT.specializations.key_for(vg, key[1], DEFAULT.backend_for(vg.kind).fp)
print(DEFAULT.specializations.probe(k2).artifact.__pdum_source__)

import math
from struct import unpack_from as _u
from pdum.dsl.kernel.registry import Out as _Out

def _tdiv(a, b):  # exact trunc division (numeric policy: C semantics)
    q = a // b
    return q + 1 if q < 0 and q * b != a else q

def _tmod(a, b):
    return a - _tdiv(a, b) * b

def kernel(staging, leaves):
    if any(isinstance(x, _Out) for x in leaves):
        raise TypeError("the python backend takes no launcher data (out= is for device targets)")
    v0 = leaves[0]
    v1 = _u('<q', staging, 40)[0]
    v2 = _u('<q', staging, 16)[0]
    v3 = v1 * v2
    v4 = _u('<q', staging, 32)[0]
    v5 = _u('<q', staging, 24)[0]
    v6 = v4 * v5
    v7 = v3 + v6
    v8 = v0[v7]
    v9 = 10.0
    v10 = v8 * v9
    return v10



The woven kernel is an ordinary scalar kernel with one extra trailing
parameter — the lane coordinate, folded into the SAME stride arithmetic
every named access already does. Nothing widened; the lazy-branch guarantee
survives untouched (each lane takes its own path — no `where` wart); and on
a GPU this parameter is launch-domain-shaped, which is where the batch axis
always wanted to live.

## Named contraction, and batching for free

The stretch goal: `matmul(A, B, i, j)` pairs the operands' UNIQUE shared
axis name (that is the whole "rules engine" — a dozen lines shaped like a
type rule), refuses ambiguity loudly, and expands to the element loop with
the trip count read from the shape SLOT — rank-generic, so new extents are
cache hits. Woven axes are excluded from pairing FIRST, which is exactly
why batching composes:

In [8]:
A = Named(np.arange(6.0).reshape(2, 3), ("row", "inner"))
B = Named(np.arange(12.0).reshape(3, 4), ("inner", "col"))


def make_mm(A, B):
    @jit()
    def cell(i, j):
        return matmul(A, B, i, j)  # noqa: F821 — 'inner' pairs BY NAME

    return cell


cell = make_mm(A, B)
got = np.array([[cell(i, j) for j in range(4)] for i in range(2)])
print("matmul == numpy @ :", np.allclose(got, A.array @ B.array))

try:
    make_mm(A, Named(np.ones((3, 4)), ("k", "col")))(0, 0)
except Exception as e:
    print("🚫 no shared name:", str(e)[:80])

rng = np.random.default_rng(7)
Ab = Named(rng.standard_normal((5, 2, 3)), ("batch", "row", "inner"))
Bb = Named(rng.standard_normal((5, 3, 4)), ("batch", "inner", "col"))
bcell = vmap(make_mm(Ab, Bb), axis="batch")  # SAME cell kernel, batch woven in
got3 = np.array([[[bcell(i, j, b) for j in range(4)] for i in range(2)] for b in range(5)])
print("batched matmul == np.matmul:", np.allclose(got3, Ab.array @ Bb.array))

matmul == numpy @ : True
🚫 no shared name: matmul needs exactly ONE shared axis name to contract; ('row', 'inner') vs ('k',
batched matmul == np.matmul: True


`batch` appears in BOTH operands, and without the woven-axes-first
exclusion it would be a contraction candidate — the pairing rule sees only
what vmap has not claimed. One batch-ignorant cell kernel; `np.matmul`
agreement per (b, i, j); zero new mechanism.

## Things to notice

- **The spike verdict**: SIMT weaving makes vmap-over-`if`/`for` a
  non-event — the 180-lines-per-region-op price was for value-widening,
  which we do not do. What is LOST: cross-lane collectives (no `psum` over
  a woven axis) — recorded, GPU-shaped, deferred.
- **One tangent engine, two doors**: `jvp` seeds tangent params; `D` seeds
  basis vectors and shares memoized tangents across calls in the same
  kernel. Custom ops join by registering a jvp rule — the transform column
  is surface-A-shaped.
- **`D` is why forward mode came first**: two inputs, many outputs is the
  shader's economics. `grad` (step 13) inverts it: transpose of this
  column, with jvp and finite differences as its oracles.
- Everything here is satellite code: 338 counted lines, zero kernel edits,
  and the walker/renderers learned nothing new — transformed kernels are
  ordinary kernels by the time a backend sees them.